![Week 4 Day 5](../assets/banners/w4d5.png)

# Week 4 Exercise: Compare Two Vision Models on CER and Cost

Asha has a budget question. *"The expensive vision model reads beautifully, but next term I'll have 200 scripts. Is the cheaper model good enough? And how would I even know?"*

That's the right question, and you now have every tool to answer it with **numbers instead of vibes**: the transcription subgraph (Day 5), CER and WER against the ground truth (Day 4), and token-based cost logging. In this exercise you'll run the same synthetic sheets through two vision models and build a small, honest comparison: accuracy, cost and speed, side by side.

### 🎯 What you'll build
1. A **harness** that transcribes the same sheets with two models (first pass only, so the comparison is fair)
2. A **scorer**: mean CER and WER per sheet against `transcripts.json`
3. A **cost model**: tokens × your provider's prices
4. A results table and a **CER vs cost** chart, and a one-paragraph recommendation for Asha

Cells marked `# ✋ TODO` are yours. Everything else is given. The notebook runs **as-is in dry-run mode** (two fake models, no key, no cost), so you can build and debug the whole harness for free, then flip to real models.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Real runs cost money: roughly 1–3 cents per sheet per model with the defaults. Start in dry-run mode, test on <b>one</b> sheet, then scale up. And make sure both models <b>accept images</b>: not every chat model is a vision model.</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("❌ OpenAI API Key not set. Most of today's cells still run offline; the few that call a model will")
    print("   show a friendly 'Missing API key' panel. See Part 5 of the setup guide, or setup/SWAP_PROVIDER.md")

In [ ]:
import json

from gradewise.config import DATA_DIR, OUTPUT_DIR, REPO_ROOT
from gradewise.console import success, warn

SYNTHETIC = DATA_DIR / "answer-sheets" / "synthetic"
SHEETS = {p.stem: p for p in sorted(SYNTHETIC.glob("student_*.pdf"))}
TRUTH_FILE = SYNTHETIC / "transcripts.json"
TRUTH = json.loads(TRUTH_FILE.read_text(encoding="utf-8")) if TRUTH_FILE.exists() else {}

if SHEETS and TRUTH:
    success("Synthetic answer sheets found", {"Folder": str(SYNTHETIC.relative_to(REPO_ROOT)), "Sheets": len(SHEETS)})
else:
    # ⚠️ Plan B: the sheets aren't there yet (run `uv run scripts/make_synthetic_sheets.py`, or `git pull`).
    #    We draw a small stand-in sheet so the rest of the lab still runs. Its text is our "ground truth".
    from PIL import Image, ImageDraw, ImageFont

    warn("data/answer-sheets/synthetic/ is missing or incomplete: using a small stand-in sheet instead.")
    standin = {
        "Q3": "Dedlock is when a process waits for a long time. Conditions: mutual exclution and circuler wait.",
        "Q4a": "When two threads use the same varible at the same time.",
    }
    try:
        font = ImageFont.truetype("C:/Windows/Fonts/Inkfree.ttf", 34)
    except OSError:
        font = ImageFont.load_default()
    pages = []
    for label, qid in [("3)", "Q3"), ("4a)", "Q4a")]:
        img = Image.new("RGB", (1240, 1754), "white")
        draw = ImageDraw.Draw(img)
        words, lines, line = f"{label} {standin[qid]}".split(), [], ""
        for word in words:
            if len(line) + len(word) > 48:
                lines.append(line)
                line = ""
            line += word + " "
        for i, text in enumerate([*lines, line]):
            draw.text((90, 140 + i * 70), text, fill=(28, 45, 140), font=font)
        pages.append(img)
    target = OUTPUT_DIR / "week4" / "student_standin.pdf"
    target.parent.mkdir(parents=True, exist_ok=True)
    pages[0].save(target, "PDF", resolution=150, save_all=True, append_images=pages[1:])
    SHEETS, TRUTH = {"student_standin": target}, {"student_standin": standin}

In [ ]:
from gradewise.schemas import ExamPaper

EXAM_FILE = DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json"
exam = ExamPaper.model_validate_json(EXAM_FILE.read_text(encoding="utf-8"))
QUESTION_IDS = [q.id for q in exam.questions]
print(f"{exam.title}: {len(QUESTION_IDS)} questions → {', '.join(QUESTION_IDS)}")

In [ ]:
def show_table(rows: list[dict]) -> None:
    """Render a list of dicts as a Markdown table (a tiny stand-in for pandas: no extra dependencies)."""
    if not rows:
        print("(no rows)")
        return
    cols = list(rows[0])

    def cell(value) -> str:
        return str(value).replace("|", "\\|").replace("\n", " ⏎ ")

    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(cell(r.get(c, "")) for c in cols) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

In [ ]:
import re
import time

from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

from gradewise.transcription import AnswerSegment, PageTranscription


class FakeVisionLLM:
    """Stands in for a vision model: its structured output replays canned PageTranscriptions.

    💡 The same trick as tests/test_transcription.py. It reads the page number from the prompt,
       because parallel branches can call it in any order. `delay` fakes the network wait.
    """

    def __init__(self, first, second=None, delay: float = 0.0):
        self.first, self.second, self.delay = first, second or {}, delay
        self.calls: list[tuple[int, bool]] = []

    def with_structured_output(self, schema, **kwargs):
        return RunnableLambda(self._respond)

    def _respond(self, messages):
        time.sleep(self.delay)
        text = messages[-1].content[0]["text"]
        number = int(re.search(r"page (\d+)", text)[1])
        second = "second, independent reading" in text
        self.calls.append((number, second))
        pages = self.second if second and number in self.second else self.first
        parsed = pages.get(number, PageTranscription(page_number=number, confidence=0.9))
        raw = AIMessage(content="", usage_metadata={"input_tokens": 1500, "output_tokens": 250, "total_tokens": 1750})
        return {"raw": raw, "parsed": parsed, "parsing_error": None}

## Part 1: The contenders

We never hard-code model names: model A is your `VISION_MODEL`, model B is `VISION_MODEL_B` from `.env` if you set one, or else your everyday `CHAT_MODEL`. To compare, say, a Claude model with a Gemini one, add `VISION_MODEL_B=provider:model` to `.env` (and that provider's key).

In [ ]:
from gradewise.config import settings
from gradewise.errors import MissingAPIKeyError
from gradewise.models import make_chat_model

MODEL_A = settings.transcription_model
MODEL_B = os.getenv("VISION_MODEL_B") or settings.chat_model
if MODEL_A == MODEL_B:
    warn("Both contenders are the same model. Set VISION_MODEL_B in .env for a real comparison.")

try:
    llm_a, llm_b = make_chat_model(MODEL_A, temperature=0), make_chat_model(MODEL_B, temperature=0)
    DRY_RUN = False
except MissingAPIKeyError as err:
    DRY_RUN = True
    warn(f"Dry run: {err}. Using two fake models, so everything below is free.")
print(f"A = {MODEL_A}\nB = {MODEL_B}\nDRY_RUN = {DRY_RUN}")

### Dry-run models

In dry-run mode, model A is a careful reader (it returns the ground truth exactly), and model B is a cheaper, sloppier one: it "helpfully" fixes spelling (Day 4's cardinal sin), skips a word now and then, and is overconfident about it. That gives your harness something real to measure.

In [ ]:
from gradewise.transcription.pdf_to_images import page_count

FIXES = {
    "Dedlock": "Deadlock",
    "exclution": "exclusion",
    "circuler": "circular",
    "proccess": "process",
    "procces": "process",
    "runing": "running",
    "kernal": "kernel",
    "somthing": "something",
    "varible": "variable",
    "Semafore": "Semaphore",
    "adress": "address",
    "Consistancy": "Consistency",
    "Durabilty": "Durability",
    "dependancys": "dependencies",
    "preemtion": "preemption",
    "conection": "connection",
}


def sloppy(text: str) -> str:
    for wrong, right in FIXES.items():
        text = re.sub(rf"\b{wrong}\b", right, text)
    words = text.split(" ")
    return " ".join(w for i, w in enumerate(words) if i % 17 != 16)  # drops every 17th word


def fake_model(sid: str, careful: bool) -> FakeVisionLLM:
    """A fake vision model for one sheet: the truth (careful) or a sloppy reading of it."""
    answers = list(TRUTH[sid].items())
    n = page_count(SHEETS[sid])
    per_page = -(-len(answers) // n)
    pages = {
        p: PageTranscription(
            page_number=p,
            confidence=0.9,
            segments=[
                AnswerSegment(label=q, text=t if careful else sloppy(t), confidence=0.9 if careful else 0.97)
                for q, t in answers[(p - 1) * per_page : p * per_page]
            ],
        )
        for p in range(1, n + 1)
    }
    return FakeVisionLLM(pages)

## Part 2: The harness

`run_sheet()` is given: it runs the transcription subgraph **without** verification (we want each model's raw first-pass quality, and a second pass would blur the comparison), and returns the sheet, the token usage the provider reported, and the wall time.

In [ ]:
from gradewise.transcription import build_transcription_graph, cer, wer  # noqa: F401 (for your TODOs)


def run_sheet(llm, sid: str, dpi: int = 200) -> dict:
    app = build_transcription_graph(llm, verify=False)
    start = time.perf_counter()
    state = app.invoke(
        {"pdf_path": str(SHEETS[sid]), "exam": exam, "student_id": sid, "dpi": dpi}, config={"max_concurrency": 4}
    )
    return {"sheet": state["sheet"], "usage": state.get("usage", []), "seconds": time.perf_counter() - start}

### Prices

Look up both models' prices (US$ per **million** tokens, input and output) on your provider's pricing page and fill them in. The defaults are the rough placeholders from `gradewise/transcription/graph.py`, with model B set at a third of the price so the dry run has something to compare.

In [ ]:
from gradewise.transcription.graph import ROUGH_PRICE_PER_M_INPUT, ROUGH_PRICE_PER_M_OUTPUT

# ✋ TODO: replace with your providers' real prices: (input $/M tokens, output $/M tokens)
PRICES = {
    MODEL_A: (ROUGH_PRICE_PER_M_INPUT, ROUGH_PRICE_PER_M_OUTPUT),
    MODEL_B: (ROUGH_PRICE_PER_M_INPUT / 3, ROUGH_PRICE_PER_M_OUTPUT / 3),
}

## Part 3: Scoring and cost

In [ ]:
def score_sheet(sheet, truth: dict[str, str]) -> dict[str, float]:
    """Mean CER and WER over the questions the student actually answered (ground truth exists)."""
    # ✋ TODO: for every question ID in `truth`, compare truth[qid] with sheet.answer_for(qid).text
    #          using cer() and wer(), and return the means. Bonus: also return the worst question.
    return {"cer": float("nan"), "wer": float("nan"), "worst": "?"}


def cost_of(usage: list[dict], model: str) -> tuple[int, int, float]:
    """(input tokens, output tokens, US$) for one sheet."""
    # ✋ TODO: add up input_tokens and output_tokens over `usage`, then price them with PRICES[model]
    return 0, 0, 0.0

In [ ]:
# 🧪 A quick smoke test on one sheet before the full run
first_sheet = next(iter(SHEETS))
smoke = run_sheet(fake_model(first_sheet, careful=False) if DRY_RUN else llm_b, first_sheet)
print(score_sheet(smoke["sheet"], TRUTH[first_sheet]), cost_of(smoke["usage"], MODEL_B))

## Part 4: The comparison

Three sheets make a fair, cheap test set: one neat, one weak speller, one messy phone photo with an injection attempt.

In [ ]:
SHEETS_TO_TEST = [s for s in ("student_03", "student_07", "student_10") if s in SHEETS] or list(SHEETS)[:3]

results = []
for label, model in (("A", MODEL_A), ("B", MODEL_B)):
    for sid in SHEETS_TO_TEST:
        llm = fake_model(sid, careful=(label == "A")) if DRY_RUN else (llm_a if label == "A" else llm_b)
        run = run_sheet(llm, sid)
        scores = score_sheet(run["sheet"], TRUTH[sid])
        tokens_in, tokens_out, dollars = cost_of(run["usage"], model)
        results.append(
            {
                "model": f"{label}: {model}",
                "sheet": sid,
                "CER": scores["cer"],
                "WER": scores["wer"],
                "worst question": scores["worst"],
                "tokens in/out": f"{tokens_in:,} / {tokens_out:,}",
                "cost $": dollars,
                "seconds": round(run["seconds"], 1),
            }
        )

show_table([{**r, "CER": f"{r['CER']:.1%}", "WER": f"{r['WER']:.1%}", "cost $": f"{r['cost $']:.4f}"} for r in results])

In [ ]:
# ✋ TODO: summarise per model: mean CER, mean WER, mean cost per sheet, and the cost of 200 scripts.

In [ ]:
# ✋ TODO: plot CER (y) against cost per sheet (x), one colour per model, each point labelled with its sheet.
#          Hint: matplotlib's ax.scatter and ax.annotate. Which corner of the chart is "best"?

### ✋ TODO: your recommendation for Asha

Write one paragraph. Which model should she use for 200 scripts, and why? Consider: where (on which sheets) do the models differ most? What kind of errors does each make (Day 4: are any of them "fixes")? How confident was each model when it was wrong? And what does an error cost in teacher time, compared with the price difference?

*Your answer here.*

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">Comparisons like this are how GradeWise's defaults were chosen, and in week 5 they become automatic: the transcription CER on the synthetic set is tracked as a <b>regression test</b> next to grading accuracy, so swapping <code>VISION_MODEL</code> in <code>.env</code> can never silently make the grader worse.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;"><b>Stretch goals.</b> 1. Add a third contender (a local Ollama vision model, if you have one: it costs $0). 2. Re-run at <code>dpi=150</code> and <code>dpi=200</code> and add DPI as a column. 3. Turn verification back on (<code>verify=True</code>) for the cheaper model: does a second pass close the gap, and what does it cost? Solution: <code>week4/solutions/week4_EXERCISE_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>